In [32]:
import pandas as pd


In [33]:
# File path to Xana Data 

EXPORT_DIR = "C:/Users/Mercy/Documents/Tendri/Snowflake Pulls/Xana/snowflake/exports/"

# inventory tables
inventory_products = pd.read_csv(EXPORT_DIR + "inventory_inventory_products.csv")
inventory_categories = pd.read_csv(EXPORT_DIR + "inventory_inventory_categories.csv")
inventory_stores = pd.read_csv(EXPORT_DIR + "inventory_stores.csv")
inventory_store_products = pd.read_csv(EXPORT_DIR + "inventory_store_products.csv")
inventory_inventory_stocks = pd.read_csv(EXPORT_DIR + "inventory_inventory_stocks.csv")
inventory_inventory_batch_product_sales = pd.read_csv(EXPORT_DIR + "inventory_inventory_batch_product_sales.csv")

# sales tables
evaluation_pos_sale_details = pd.read_csv(EXPORT_DIR + "evaluation_pos_sale_details.csv")

# finance tables - corporate
finance_invoices = pd.read_csv(EXPORT_DIR + "finance_invoices.csv")
finance_invoice_items = pd.read_csv(EXPORT_DIR + "finance_invoice_items.csv")
finance_invoice_payments = pd.read_csv(EXPORT_DIR + "finance_invoice_payments.csv")

# finance tables - retail
finance_evaluation_payments = pd.read_csv(EXPORT_DIR + "finance_evaluation_payments.csv")
finance_evaluation_payments_details = pd.read_csv(EXPORT_DIR + "finance_evaluation_payments_details.csv")


# customer tables
customers = pd.read_csv(EXPORT_DIR + "reception_patients.csv")

In [34]:
# lowercase all column names
for df in [
    inventory_products, inventory_categories, inventory_stores,
    inventory_store_products, inventory_inventory_stocks,
    inventory_inventory_batch_product_sales, evaluation_pos_sale_details,
    finance_invoices, finance_invoice_items, finance_invoice_payments,
    finance_evaluation_payments, finance_evaluation_payments_details,
    customers
]:
    df.columns = df.columns.str.lower()


In [36]:
merged = pd.merge(
    evaluation_pos_sale_details,
    inventory_store_products,
    left_on  = "store_product_id",
    right_on = "id",
    how      = "left",
    suffixes = ("_sale", "_products")
)

In [37]:
print(f"evaluation_pos_sale_details rows : {len(evaluation_pos_sale_details):,}")
print(f"merged table rows                : {len(merged):,}")
print(f"rows lost in join                : {len(evaluation_pos_sale_details) - len(merged):,}")
print(f"match rate                       : {len(merged) / len(evaluation_pos_sale_details):.1%}")

evaluation_pos_sale_details rows : 154,226
merged table rows                : 154,226
rows lost in join                : 0
match rate                       : 100.0%


In [38]:
with_store_id    = merged[merged["store_id"].notna()]
without_store_id = merged[merged["store_id"].isna()]

print(f"Rows with store_id    : {len(with_store_id):,}")
print(f"Rows without store_id : {len(without_store_id):,}")


Rows with store_id    : 153,712
Rows without store_id : 514


In [39]:
merged2 = pd.merge(
    merged,
    inventory_stores,
    left_on  = "store_id",
    right_on = "id",
    how      = "inner",
    suffixes = ("", "_store")
)

print(f"merged rows  : {len(merged):,}")
print(f"merged2 rows : {len(merged2):,}")
print(f"rows lost    : {len(merged) - len(merged2):,}")
print(f"match rate   : {len(merged2) / len(merged):.1%}")



merged rows  : 154,226
merged2 rows : 154,226
rows lost    : 0
match rate   : 100.0%


In [27]:
merged2.to_csv(EXPORT_DIR + "pos_store_sales_details.csv", index=False)
print("\nSaved: pos_store_sales_details.csv")


Saved: pos_store_sales_details.csv


In [ ]:
checking = pd.read_csv(EXPORT_DIR + "pos_store_sales_details.csv")
checking.count()

id_sale                      154226
sale_id                      154226
pos_id                       154226
item_id                      154226
store_product_id             154226
service_id                   154226
name                         154226
type                         154226
status                       154226
prescription_note                 0
quantity_sale                154226
price                        154226
amount                       154226
discount                     154226
round_up                     154226
created_at_sale              154226
updated_at_sale              154226
id_products                  153712
product_id                   153712
store_id                     153712
bar_code                          0
lot_no                         9225
quantity_products            153712
re_order_level                 9393
selling_price                153712
discount_price                    0
lower_limit_price             83685
lower_limit_percentage      

In [41]:
merged_customers = pd.merge(
    inventory_inventory_batch_product_sales,
    customers,
    left_on  = "patient",
    right_on = "id",
    how      = "inner",
    suffixes = ("_sale", "_customer")
)

print(f"batch_sales rows          : {len(inventory_inventory_batch_product_sales):,}")
print(f"merged rows               : {len(merged_customers):,}")
print(f"rows lost                 : {len(inventory_inventory_batch_product_sales) - len(merged_customers):,}")
print(f"match rate                : {len(merged_customers) / len(inventory_inventory_batch_product_sales):.1%}")


batch_sales rows          : 61,449
merged rows               : 33,293
rows lost                 : 28,156
match rate                : 54.2%


In [44]:
merged_sales = pd.merge(
    inventory_inventory_batch_product_sales,
    evaluation_pos_sale_details,
    left_on  = "id",
    right_on = "sale_id",
    how      = "inner",
    suffixes = ("_batch_sale", "_sales_details")
)

print(f"batch_sales rows          : {len(evaluation_pos_sale_details):,}")
print(f"merged rows               : {len(merged_sales):,}")
print(f"rows lost                 : {len(evaluation_pos_sale_details) - len(merged_sales):,}")
print(f"match rate                : {len(merged_sales) / len(evaluation_pos_sale_details):.1%}")

batch_sales rows          : 154,226
merged rows               : 154,202
rows lost                 : 24
match rate                : 100.0%


In [49]:
inventory_store_products.nunique()

id                          13721
product_id                   7625
store_id                        8
bar_code                        0
lot_no                       2572
quantity                      554
re_order_level               1330
selling_price                 947
discount_price                  0
lower_limit_price            1151
lower_limit_percentage         87
insurance_price                 1
total_insurance_price         959
unit_cost                    5487
total_cost                      1
total_cash_price               40
bulk_pay_quantity               1
bulk_percentage_discount        1
expiry_date                   327
created_at                  12577
updated_at                   4929
deleted_at                      0
dtype: int64

# INVENTORY

In [20]:

#merge inventory products and inventory categories
inventory_products_categories = pd.merge(inventory_products, inventory_categories, left_on='category', right_on='id')
inventory_products_categories.head()

,id_x,code_x,item_no,name_x,description,bar_code,category,department,unit,tax_category,...,id_y,code_y,name_y,parent,parent_id,cash_markup,credit_markup,created_at_y,updated_at_y,deleted_at_y
0,38192.0,SOD0023,SOD0023,Morning Fresh Dwl Orange 400Ml,Morning Fresh Dwl Orange 400Ml,6161102002881,1587.0,NaN,34702.0,5.0,...,1587,NaN,DETERGENT,NaN,1573.0,0.0,0.0,2025-09-11 10:09:55,2025-11-17 10:06:16,NaN
1,38753.0,CHR0031,CHR0031,Atorvastatin 40Mg Tablets 30'S,Atorvastatin 40Mg Tablets 30'S,NaN,1616.0,NaN,34707.0,7.0,...,1616,NaN,CHRONIC,NaN,1648.0,0.0,0.0,2025-09-11 10:09:57,2025-11-17 10:06:14,NaN
2,38754.0,CHR0032,CHR0032,Atstat 20Mg Tablets 30'S,Atstat 20Mg Tablets 30'S,NaN,1616.0,NaN,34707.0,7.0,...,1616,NaN,CHRONIC,NaN,1648.0,0.0,0.0,2025-09-11 10:09:57,2025-11-17 10:06:14,NaN
3,39540.0,PGI0047,PGI0047,Agycin 200Mg/5Ml Suspension 15Ml,Agycin 200Mg/5Ml Suspension 15Ml,NaN,1648.0,NaN,34701.0,7.0,...,1648,NaN,GENERAL,NaN,1718.0,0.0,0.0,2025-09-11 10:10:00,2025-11-17 10:06:18,NaN
4,39541.0,PGI0048,PGI0048,Agycin 250Mg Tablets 6'S,Agycin 250Mg Tablets 6'S,NaN,1648.0,NaN,34707.0,7.0,...,1648,NaN,GENERAL,NaN,1718.0,0.0,0.0,2025-09-11 10:10:00,2025-11-17 10:06:18,NaN


# PIVOT FINANCE EVALUATION PAYMENT

In [54]:
merged_payments = pd.merge(
    finance_evaluation_payments,finance_evaluation_payments_details,
    left_on  = "id", 
    right_on = "payment_id",
    how      = "inner",
    suffixes = ("_payment", "_payment_details")
)


In [ ]:
payments_with_sale = finance_evaluation_payments.merge(
    inventory_inventory_batch_product_sales[['id', 'patient', 'sale_id']],
    left_on='patient_id',
    right_on='patient',
    how='left'
)

In [55]:
payment_cols = [
    'card_amount',
    'mpesa_amount', 
    'jambopay_amount',
    'pesa_pal_card_amount',
    'pesa_pal_mpesa_amount',
    'cheque_amount',
    'giftcard_amount'
]

print(merged_payments.columns.tolist())


['id_payment', 'receipt', 'receipt_prefix', 'patient_id', 'patient_name', 'patient_no', 'user_id', 'user_name', 'visit_id_payment', 'sale', 'sale_id', 'invoice_id', 'facility_id', 'scu_id', 'transaction_id', 'status', 'payment_mode', 'etims_receipt_sign', 'discount_reason', 'amount_payment', 'total_amount_was', 'total_cash_payment', 'payable_amount', 'cash_amount', 'card_amount', 'mpesa_amount', 'jambopay_amount', 'pesa_pal_card_amount', 'pesa_pal_mpesa_amount', 'cheque_amount', 'giftcard_amount', 'loyalty_amount', 'points_amount', 'voucher_amount', 'waiver_amount', 'discount_payment', 'deposit', 'dispensing', 'change', 'vat_amount_payment', 'patient_account_amount', 'mpesa_fix_status', 'mpesa_fix_at', 'mpesa_fix_confidence', 'mpesa_fix_old_txn_id', 'mpesa_fix_suggested_txn_id', 'receipt_date', 'created_at_payment', 'updated_at_payment', 'deleted_at_payment', 'id_payment_details', 'payment_id', 'prescription_id', 'visit_id_payment_details', 'store_id', 'consumable_id', 'investigation_i